# 02 — Clean & Build the Recommender Data Layer

Cleans the Sephora data, builds the product-level tables the chatbot/recommender will query, and draws EDA charts inline.

**Outputs** (in `data/processed/`):
- `products_clean.parquet` — cleaned skincare catalogue
- `product_review_agg.parquet` — per-product review stats
- `recommender_products.parquet` — merged layer the chatbot queries
- `product_skintone_matrix.parquet` — fairness matrix (product × skin_tone)

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

if os.path.basename(os.getcwd()).lower() == 'notebooks':
    os.chdir('..')
print('cwd:', os.getcwd())

RAW  = os.path.join('data', 'sephora', 'archive')
PROC = os.path.join('data', 'processed')
FIG  = os.path.join('reports', 'figures')
os.makedirs(PROC, exist_ok=True)
os.makedirs(FIG, exist_ok=True)
CHUNK = 100_000

## 1. Clean products (skincare only)

In [ ]:
prod = pd.read_csv(os.path.join(RAW, 'product_info.csv'))
prod = prod[prod['primary_category'].astype(str).str.contains('Skincare', case=False, na=False)].copy()

for c in ['price_usd','rating','reviews','loves_count']:
    prod[c] = pd.to_numeric(prod[c], errors='coerce')

prod['ingredient_count'] = (
    prod['ingredients'].astype(str).where(prod['ingredients'].notna(), '')
    .apply(lambda s: 0 if s.strip()=='' else len([x for x in s.split(',') if x.strip()])))

keep = ['product_id','product_name','brand_name','price_usd','rating','reviews',
        'loves_count','ingredients','ingredient_count','highlights',
        'primary_category','secondary_category','tertiary_category','size','out_of_stock','new']
prod = prod[[c for c in keep if c in prod.columns]]
prod.to_parquet(os.path.join(PROC, 'products_clean.parquet'), index=False)
print(f'skincare products kept: {len(prod):,}')
prod.head(3)

## 2. Aggregate reviews per product (chunked)
Builds review stats, a per-`skin_type` fit table, and a per-`skin_tone` fairness matrix.

In [ ]:
base_parts, stype_parts, stone_parts = [], [], []
usecols = ['rating','is_recommended','review_text','skin_tone','skin_type','product_id']

for f in sorted(glob.glob(os.path.join(RAW, 'reviews_*.csv'))):
    for ch in pd.read_csv(f, usecols=usecols, chunksize=CHUNK, low_memory=False):
        ch['rating'] = pd.to_numeric(ch['rating'], errors='coerce')
        ch['is_recommended'] = pd.to_numeric(ch['is_recommended'], errors='coerce')
        ch['is_positive'] = (ch['rating'] >= 4).astype(int)          # sentiment proxy
        ch['text_len'] = ch['review_text'].astype(str).str.len()

        base_parts.append(ch.groupby('product_id').agg(
            n_reviews=('rating','size'), rating_sum=('rating','sum'), rating_cnt=('rating','count'),
            rec_sum=('is_recommended','sum'), rec_cnt=('is_recommended','count'),
            pos_sum=('is_positive','sum'), textlen_sum=('text_len','sum')))

        stype_parts.append(ch.dropna(subset=['skin_type'])
            .groupby(['product_id','skin_type']).size().rename('n').reset_index())
        stone_parts.append(ch.dropna(subset=['skin_tone'])
            .groupby(['product_id','skin_tone']).size().rename('n').reset_index())
    print('done:', os.path.basename(f))
print('aggregation complete')

In [ ]:
# combine base stats
base = pd.concat(base_parts).groupby(level=0).sum()
base['mean_rating']     = base['rating_sum'] / base['rating_cnt']
base['pct_recommended'] = base['rec_sum']    / base['rec_cnt']
base['pct_positive']    = base['pos_sum']    / base['rating_cnt']
base['avg_text_len']    = base['textlen_sum']/ base['n_reviews']
agg = base[['n_reviews','mean_rating','pct_recommended','pct_positive','avg_text_len']].reset_index()
agg.to_parquet(os.path.join(PROC, 'product_review_agg.parquet'), index=False)
print(f'products with reviews: {len(agg):,}')
agg.sort_values('n_reviews', ascending=False).head()

In [ ]:
# per skin_type wide table
stype = (pd.concat(stype_parts).groupby(['product_id','skin_type'])['n'].sum().reset_index()
         .pivot(index='product_id', columns='skin_type', values='n').fillna(0))
stype.columns = [f'reviews_{c}' for c in stype.columns]
stype = stype.reset_index()

# per skin_tone fairness matrix
stone = (pd.concat(stone_parts).groupby(['product_id','skin_tone'])['n'].sum().reset_index()
         .pivot(index='product_id', columns='skin_tone', values='n').fillna(0).astype(int))
stone.to_parquet(os.path.join(PROC, 'product_skintone_matrix.parquet'))
print('skin_tone matrix:', stone.shape)
stone.head(3)

## 3. Merge into the recommender layer

In [ ]:
rec = (prod.merge(agg, on='product_id', how='left')
           .merge(stype, on='product_id', how='left'))
for c in [c for c in rec.columns if c.startswith('reviews_')]:
    rec[c] = rec[c].fillna(0).astype(int)
rec.to_parquet(os.path.join(PROC, 'recommender_products.parquet'), index=False)
print('recommender layer:', rec.shape)
rec.head(3)

## 4. Charts (inline + saved to reports/figures/)

In [ ]:
# price distribution
ax = prod['price_usd'].clip(upper=300).plot(kind='hist', bins=40, figsize=(7,4), color='#c98a9b')
ax.set_title('Skincare product price (USD, capped at 300)'); ax.set_xlabel('price_usd')
plt.tight_layout(); plt.savefig(os.path.join(FIG,'price_distribution.png'), dpi=120); plt.show()

In [ ]:
# skin_tone representation (fairness headline)
tone_totals = stone.sum(axis=0).sort_values(ascending=False)
ax = tone_totals.plot(kind='bar', figsize=(8,4), color='#8a6ea3')
ax.set_title('Review counts by reviewer skin_tone (representation bias)'); ax.set_ylabel('reviews')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(FIG,'skin_tone_representation.png'), dpi=120); plt.show()

In [ ]:
# skin_type mix
type_totals = stype.drop(columns='product_id').sum().rename(lambda x: x.replace('reviews_',''))
ax = type_totals.sort_values(ascending=False).plot(kind='bar', figsize=(6,4), color='#6ea38a')
ax.set_title('Review counts by reviewer skin_type'); ax.set_ylabel('reviews')
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(os.path.join(FIG,'skin_type_mix.png'), dpi=120); plt.show()

In [ ]:
# mean rating vs review volume (credibility view)
m = agg[agg['n_reviews'] >= 5]
plt.figure(figsize=(7,4))
plt.scatter(m['n_reviews'], m['mean_rating'], s=8, alpha=0.3, color='#c98a9b')
plt.xscale('log'); plt.title('Product mean rating vs #reviews')
plt.xlabel('n_reviews (log)'); plt.ylabel('mean_rating')
plt.tight_layout(); plt.savefig(os.path.join(FIG,'rating_vs_volume.png'), dpi=120); plt.show()

In [ ]:
# fairness snapshot
deep_like  = [c for c in stone.columns if c in ['deep','rich','dark','ebony']]
light_like = [c for c in stone.columns if c in ['fair','fairLight','light','lightMedium','porcelain']]
d = int(stone[deep_like].sum().sum()); l = int(stone[light_like].sum().sum()); tot = int(stone.sum().sum())
print(f'light-group reviews: {l:,} ({100*l/tot:.1f}%)')
print(f'deep-group reviews : {d:,} ({100*d/tot:.1f}%)')
print('\nThis imbalance is the representation-bias evidence for the fairness chapter.')